# Week 36, Day 5: Brain-computer interfaces: decoding in closed loop

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Rotation in closed loop

Complete the loop inside run(): compute the intended velocity (8 cm/s towards the target, scaled by min(1, distance / 3)), generate spikes from the rotated intention, decode a velocity with the weights B (spikes plus a constant) and move the cursor by velocity times dt.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2021)
dt, N = 0.05, 40
pd = r.uniform(0, 2 * np.pi, N); base = r.uniform(5, 15, N); gain = r.uniform(0.5, 1.5, N)
U = np.c_[np.cos(pd), np.sin(pd)] * gain[:, None]
spikes = lambda v: r.poisson(np.maximum(base + v @ U.T, 0) * dt)

Vc = r.normal(0, 8, (3000, 2)); Zc = spikes(Vc)
fit = lambda Z, V: np.linalg.lstsq(np.c_[Z, np.ones(len(Z))], V, rcond=None)[0]
B = fit(Zc, Vc)

def run(B, trials=60, rot_deg=0, record=False):
    a = np.radians(rot_deg); R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    times, rec, paths = [], [], []
    for _ in range(trials):
        ang = r.uniform(0, 2 * np.pi); tgt = 10 * np.array([np.cos(ang), np.sin(ang)])
        c, t, path = np.zeros(2), 0, [np.zeros(2)]
        while t < 200:
            d = tgt - c; dist = np.linalg.norm(d)
            if dist < 1.5:
                break
            ...
            t += 1; path.append(c.copy())
            if record:
                rec.append((z, 8 * d / dist))
        times.append(t * dt); paths.append(np.array(path))
    return np.array(times), rec, paths

times, fail = {}, {}
for rot in (0, 30, 60):
    tm, _, paths = run(B, rot_deg=rot)
    times[rot], fail[rot] = np.median(tm), np.mean(tm >= 10)
    print(f"rotation {rot:2d} deg: median time {times[rot]:.2f} s, failures {fail[rot]:.0%}")
    plt.plot(*paths[0].T, label=f"{rot} deg")
plt.axis("equal"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_all_reach():
    "with rotations up to 60 degrees, every trial still reaches the target within 10 s"
    assert all(fail[rot] == 0 for rot in fail)

def test_slower():
    "a 60 degree rotation makes trials at least 50% slower than no rotation"
    assert times[60] > 1.5 * times[0]

test_all_reach()
test_slower()
print("All checks passed")

### Exercise: Refitting to intention

Run 80 trials with the old decoder under a 45° rotation, recording each bin's spikes and the intended velocity (8 cm/s towards the target). Fit a new decoder to these pairs and measure the median trial time with it.

In [ ]:
tm, rec, _ = run(B, trials=80, rot_deg=45, record=True)
before = np.median(tm)
Zr = np.array([z for z, _ in rec]); Vr = np.array([v for _, v in rec])
B2 = ...
after = np.median(run(B2, rot_deg=45)[0])
print(f"45 deg rotation: median time {before:.2f} s before refitting, {after:.2f} s after ({len(rec)} bins of closed-loop data)")

In [ ]:
import numpy as np
def test_faster():
    "after refitting, trials with a 45 degree rotation are at least 15% faster"
    assert after < 0.85 * before

test_faster()
print("All checks passed")

### Exercise: Straight lines against curves

Write classify(templates): average 10 training trials per class to make a template of spike counts (20 bins by 40 neurons), then classify 10 new trials per class by the nearest template in squared distance, and return the fraction correct.

In [ ]:
S, K = 20, 26
t = np.linspace(0, 1, S)
straight = [np.outer(np.ones(S), [np.cos(a), np.sin(a)]) * 6 for a in np.linspace(0, 2 * np.pi, K, endpoint=False)]

def curvy():
    c = r.normal(size=(4, 2))
    v = sum(np.outer(np.sin((k + 1) * np.pi * t + r.uniform(0, np.pi)), c[k]) for k in range(4))
    return 6 * v / np.sqrt((v ** 2).sum(1).mean())

curved = [curvy() for _ in range(K)]

def trial(v):
    w = np.clip(t + r.normal(0, 0.12) * np.sin(np.pi * t), 0, 1)        # random time warping
    vv = np.array([np.interp(w, t, v[:, i]) for i in range(2)]).T
    return r.poisson(np.maximum(8 + vv @ U.T, 0) * dt)

def classify(templates):
    ...

acc = {"straight": classify(straight), "curved": classify(curved)}
dmin = {}
for name, T_ in (("straight", straight), ("curved", curved)):
    M = np.array([v.ravel() for v in T_]); Dm = ((M[:, None] - M[None]) ** 2).sum(-1); np.fill_diagonal(Dm, np.inf)
    dmin[name] = Dm.min()
print({k: round(float(v), 2) for k, v in acc.items()}, "accuracy;  closest pair distance", {k: round(float(v)) for k, v in dmin.items()})

In [ ]:
import numpy as np
def test_separable():
    "curved trajectories are classified at least twice as accurately as straight ones"
    assert acc["curved"] > 2 * acc["straight"]

def test_distance():
    "the smallest distance between curved templates is at least 4 times that between straight ones"
    assert dmin["curved"] > 4 * dmin["straight"]

test_separable()
test_distance()
print("All checks passed")